# Grzybowa Mapa: trening własnego detektora grzybów (YOLO)

**Jak uruchomić:**
1. Menu **Środowisko wykonawcze → Zmień typ środowiska → T4 GPU** → Zapisz.
2. Menu **Środowisko wykonawcze → Uruchom wszystko**.
3. Po ok. 40–70 min przeglądarka sama pobierze pliki **`grzyb-yolo.onnx`** i **`grzyb-yolo.pt`**. Wrzuć je do folderu `tiktok` (albo wyślij w czacie), a podłączę go do radaru.

Dane: zdjęcia z **Open Images V7** (Google), klasa *Mushroom* z ręcznie zaznaczonymi ramkami (adnotacje CC BY 4.0, zdjęcia CC BY 2.0), plus zdjęcia drzew i roślin bez grzybów jako „tło”, żeby radar nie mylił liści i kory z grzybami.

Model: **YOLO11n** (Ultralytics, licencja AGPL-3.0; kod strony i tak jest publiczny na GitHubie).

In [ ]:
# Instalacja. Jeśli biblioteki dopiero się instalują, sesja sama się zrestartuje (Pillow po aktualizacji wymaga restartu);
# wtedy po prostu kliknij jeszcze raz „Uruchom wszystko”.
import importlib.util, subprocess, sys, os
if importlib.util.find_spec("fiftyone") is None or importlib.util.find_spec("ultralytics") is None:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "ultralytics", "fiftyone"], check=True)
    print("Zainstalowano biblioteki, restartuję sesję. Potem kliknij ponownie „Uruchom wszystko”.")
    os.kill(os.getpid(), 9)
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "BRAK – włącz T4 GPU w ustawieniach!")

## 1. Pobranie danych z Open Images

In [ ]:
import fiftyone as fo, fiftyone.zoo as foz
from fiftyone import ViewField as F

def grzyby(split, n=None):
    ds = foz.load_zoo_dataset("open-images-v7", split=split, label_types=["detections"], classes=["Mushroom"],
                              max_samples=n, dataset_name=f"grzyby-{split}", shuffle=True, seed=7)
    return ds.filter_labels("ground_truth", F("label") == "Mushroom", only_matches=True)

def tlo(split, n):
    ds = foz.load_zoo_dataset("open-images-v7", split=split, label_types=["detections"], classes=["Tree", "Plant", "Leaf"],
                              max_samples=n, dataset_name=f"tlo-{split}", shuffle=True, seed=7)
    # zostawiamy tylko zdjęcia bez żadnego grzyba i czyścimy ramki (to jest czyste tło)
    ds = ds.filter_labels("ground_truth", F("label") == "Mushroom", only_matches=False)
    ds = ds.match(F("ground_truth.detections").length() == 0)
    return ds

pos_tr, pos_va = grzyby("train"), grzyby("validation")
neg_tr, neg_va = tlo("train", 900), tlo("validation", 150)
print("grzyby train/val:", len(pos_tr), len(pos_va), " tło train/val:", len(neg_tr), len(neg_va))

## 2. Eksport do formatu YOLO

In [ ]:
import os, shutil
OUT = "/content/grzyby_yolo"; shutil.rmtree(OUT, ignore_errors=True)
for view, split in [(pos_tr, "train"), (neg_tr, "train"), (pos_va, "val"), (neg_va, "val")]:
    view.export(export_dir=OUT, dataset_type=fo.types.YOLOv5Dataset, label_field="ground_truth",
                classes=["Mushroom"], split=split)
open(f"{OUT}/dataset.yaml", "w").write(f"path: {OUT}\ntrain: images/train\nval: images/val\nnames:\n  0: grzyb\n")
for s in ["train", "val"]:
    print(s, len(os.listdir(f"{OUT}/images/{s}")), "zdjęć,", len(os.listdir(f"{OUT}/labels/{s}")), "plików z ramkami")

## 2a. (opcjonalnie) Twoje własne zdjęcia
Jeśli masz zdjęcia z zaznaczonymi grzybami w formacie YOLO (np. z Roboflow lub Label Studio, eksport „YOLOv8”), wgraj zip w panelu plików po lewej jako `moje.zip` i uruchom tę komórkę. Bez pliku komórka nic nie robi.

In [ ]:
import zipfile, glob
if os.path.exists("/content/moje.zip"):
    zipfile.ZipFile("/content/moje.zip").extractall("/content/moje")
    n = 0
    for img in glob.glob("/content/moje/**/images/**/*.*", recursive=True):
        lab = img.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
        dst = f"moje_{n:05d}"; ext = img.rsplit(".", 1)[1]
        shutil.copy(img, f"{OUT}/images/train/{dst}.{ext}")
        if os.path.exists(lab):
            # wszystkie klasy z Twojego zbioru traktujemy jako „grzyb”
            rows = [("0 " + l.split(" ", 1)[1]) for l in open(lab).read().strip().splitlines() if l.strip()]
            open(f"{OUT}/labels/train/{dst}.txt", "w").write("\n".join(rows))
        n += 1
    print("dodano", n, "Twoich zdjęć")
else:
    print("brak moje.zip – pomijam")

## 3. Trening
Domyślnie **douczamy obecny model** ze strony (`grzyb-yolo.pt`): krócej (ok. 20–30 min) i z małym krokiem uczenia, żeby nie zapomniał tego, co już umie. Dane z Open Images zostają w zestawie z tego samego powodu.
Żeby uczyć od zera, ustaw `OD_ZERA = True`.

In [ ]:
import urllib.request
from ultralytics import YOLO
OD_ZERA = False
START = "yolo11n.pt"
if not OD_ZERA:
    try:
        urllib.request.urlretrieve("https://grzybowamapa.com.pl/models/grzyb-yolo.pt", "/content/start.pt"); START = "/content/start.pt"
        print("Douczam obecny model ze strony")
    except Exception as e:
        print("Nie udało się pobrać obecnego modelu, uczę od zera:", e)
douczanie = START != "yolo11n.pt"
model = YOLO(START)
model.train(data=f"{OUT}/dataset.yaml", imgsz=416, epochs=40 if douczanie else 80, patience=15 if douczanie else 20, batch=32,
            lr0=0.002 if douczanie else 0.01, warmup_epochs=1 if douczanie else 3,
            hsv_h=0.02, degrees=10, fliplr=0.5, mosaic=1.0, close_mosaic=10,
            project="/content/runs", name="grzyb", exist_ok=True, plots=True)

## 4. Wynik i eksport do przeglądarki

In [ ]:
best = YOLO("/content/runs/grzyb/weights/best.pt")
m = best.val(data=f"{OUT}/dataset.yaml", imgsz=416)
print(f"mAP50 = {m.box.map50:.3f}   precyzja = {m.box.mp:.3f}   czułość = {m.box.mr:.3f}")
path = best.export(format="onnx", imgsz=416, opset=12, simplify=True, dynamic=False)
shutil.copy(path, "/content/grzyb-yolo.onnx"); print(os.path.getsize("/content/grzyb-yolo.onnx") / 1e6, "MB")

In [ ]:
from IPython.display import Image, display
display(Image("/content/runs/grzyb/val_batch0_pred.jpg", width=900))  # podgląd: ramki modelu na zdjęciach testowych
from google.colab import files
shutil.copy("/content/runs/grzyb/weights/best.pt", "/content/grzyb-yolo.pt")
files.download("/content/grzyb-yolo.onnx")   # dla przeglądarki (radar)
files.download("/content/grzyb-yolo.pt")     # pełne wagi: od nich startuje kolejne douczanie